# Section 0 — What Does the Encoder Actually Output?
### Subject-level deep dive: raw embeddings, statistics, and visualizations

**Confirmed naming pattern:**
- `filename` = `{uuid}_{participantID}_{segment}.flac`
- `original_stem` = `{uuid}_{participantID}` — matches all 3 segments per subject
- `seq_path` → raw `[T, 768]` encoder output (one row per ~20 ms chunk)
- `embedding_path` → mean-pooled `[768]` final fingerprint

In [1]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics.pairwise import cosine_similarity

BASE      = os.path.abspath(os.getcwd())
META_PATH = os.path.join(BASE, "metadata.csv")
OUT_DIR   = os.path.join(BASE, "results", "subject_stories")
os.makedirs(OUT_DIR, exist_ok=True)

SEGMENTS   = ["early", "middle", "late"]
SEG_COLORS = {"early": "#4CAF50", "middle": "#FF9800", "late": "#9C27B0"}

meta = pd.read_csv(META_PATH)
print(f"Metadata loaded: {len(meta)} rows")
print(f"Columns: {list(meta.columns)}")

Metadata loaded: 3741 rows
Columns: ['filename', 'class', 'segment', 'original_stem', 'embedding_path', 'seq_path']


In [2]:
def get_complete_subjects(df, cls):
    sub    = df[df["class"] == cls]
    counts = sub.groupby("original_stem")["segment"].nunique()
    return list(counts[counts == 3].index)

pd_stems = get_complete_subjects(meta, "PD")
hc_stems = get_complete_subjects(meta, "HC")
print(f"Complete PD subjects (all 3 segments): {len(pd_stems)}")
print(f"Complete HC subjects (all 3 segments): {len(hc_stems)}")

selected_pd = sorted(pd_stems)[:3]
selected_hc = sorted(hc_stems)[:3]
selected    = [(s, "PD") for s in selected_pd] + [(s, "HC") for s in selected_hc]

print()
print("=" * 70)
print("Selected subjects for analysis:")
print("=" * 70)
print(f"  {'#':<3} {'Class':<6} original_stem")
print("  " + "-" * 65)
for i, (stem, cls) in enumerate(selected, 1):
    print(f"  {i:<3} {cls:<6} {stem}")

Complete PD subjects (all 3 segments): 338
Complete HC subjects (all 3 segments): 909

Selected subjects for analysis:
  #   Class  original_stem
  -----------------------------------------------------------------
  1   PD     000c901b-049b-41e1-b238-eae9c209edaa_6023402
  2   PD     01c1c8cc-cc5b-48a4-b1b8-431065bf780f_6013793
  3   PD     024dc78c-80c8-4d52-8600-0101d0fa2c83_6012162
  4   HC     003d12eb-7a9c-4ab2-aed8-e6097c5d5c38_6013609
  5   HC     0055c363-8a10-4081-91e2-dae9d082752c_6011857
  6   HC     00cc4230-4c28-4546-b353-e3ae13a7f0b0_6001885


---
## What Is a HuBERT Embedding?

Before we look at the numbers, here is a plain-English explanation
of what the encoder actually produces.

In [3]:
print("""
WHAT IS A WAV2VEC2.0 EMBEDDING?
================================
When you give HuBERT a voice clip, it does NOT give you back
audio. It gives you back NUMBERS — specifically a big table of numbers.

Think of it like this:
  - You speak for 3 seconds (early segment)
  - HuBERT listens and says:
    "This voice has these 768 characteristics"
  - It writes down 768 numbers describing your voice

But actually it does this for EVERY ~20ms chunk of your audio.
So for a 3-second clip:
  3 seconds / 0.02 seconds = ~150 time steps
  Each time step -> 768 numbers
  Final output shape: [150, 768]  (115,200 numbers total)

We then AVERAGE all 150 rows -> get 1 row of 768 numbers.
This 1x768 vector = the fingerprint of that voice segment.

The 768 numbers do NOT have labels like pitch or tremor.
They are abstract features learned from millions of hours of speech.
""")


WHAT IS A WAV2VEC2.0 EMBEDDING?
When you give HuBERT a voice clip, it does NOT give you back
audio. It gives you back NUMBERS — specifically a big table of numbers.

Think of it like this:
  - You speak for 3 seconds (early segment)
  - HuBERT listens and says:
    "This voice has these 768 characteristics"
  - It writes down 768 numbers describing your voice

But actually it does this for EVERY ~20ms chunk of your audio.
So for a 3-second clip:
  3 seconds / 0.02 seconds = ~150 time steps
  Each time step -> 768 numbers
  Final output shape: [150, 768]  (115,200 numbers total)

We then AVERAGE all 150 rows -> get 1 row of 768 numbers.
This 1x768 vector = the fingerprint of that voice segment.

The 768 numbers do NOT have labels like pitch or tremor.
They are abstract features learned from millions of hours of speech.



In [4]:
def cos_sim(a, b):
    return float(cosine_similarity(a.reshape(1, -1), b.reshape(1, -1))[0, 0])

def short(stem, n=30):
    return stem[:n] + "..." if len(stem) > n else stem

In [5]:
summary_rows = []

for subj_idx, (orig_stem, cls) in enumerate(selected, 1):
    print(f"\nProcessing subject {subj_idx}/6: {orig_stem}")
    print("=" * 70)

    subj_rows = meta[meta["original_stem"] == orig_stem]

    seq = {}
    emb = {}
    for seg in SEGMENTS:
        r         = subj_rows[subj_rows["segment"] == seg].iloc[0]
        seq[seg]  = np.load(r["seq_path"])          # (T, 768)
        emb[seg]  = np.load(r["embedding_path"])    # (768,)
        if emb[seg].ndim == 2:
            emb[seg] = emb[seg].mean(axis=0)

    display_name = short(orig_stem)

    # ── STEP A ────────────────────────────────────────────────────────────
    print(f"{'=' * 50}")
    print(f"SUBJECT REPORT: {orig_stem}  |  Class: {cls}")
    print(f"{'=' * 50}")
    print()
    print("STEP A: RAW SHAPE BEFORE AVERAGING")
    print("Raw encoder output (before averaging):")
    print(f"  early  segment -> shape: [{seq['early'].shape[0]}  time_steps x {seq['early'].shape[1]} features]")
    print(f"  middle segment -> shape: [{seq['middle'].shape[0]} time_steps x {seq['middle'].shape[1]} features]")
    print(f"  late   segment -> shape: [{seq['late'].shape[0]}  time_steps x {seq['late'].shape[1]} features]")
    print()
    print("This means:")
    print(f"  early : encoder listened {seq['early'].shape[0]}  times (every ~20ms), 768 numbers each time")
    print(f"  middle: encoder listened {seq['middle'].shape[0]} times")
    print(f"  late  : encoder listened {seq['late'].shape[0]}  times")
    print()

    # ── STEP B ────────────────────────────────────────────────────────────
    raw_e = seq["early"]
    print("STEP B: RAW OUTPUT — first 5 time steps x first 10 features (early segment):")
    print("(Each row = one ~20ms chunk of voice. Each column = one feature.)")
    print()
    header = "        " + "  ".join([f"feat_{j:<4}" for j in range(10)])
    print(header)
    for t in range(min(5, raw_e.shape[0])):
        vals = "  ".join([f"{raw_e[t, j]:>8.3f}" for j in range(10)])
        print(f"time_{t}: {vals}")
    print()
    print("Notice: numbers can be positive or negative, roughly between -5 and +5.")
    print("These are NOT pitch/loudness/tremor. They are abstract neural features.")
    print()

    # ── STEP C ────────────────────────────────────────────────────────────
    print("STEP C: AVERAGED EMBEDDING (final fingerprint) — first 20 of 768 numbers:")
    for seg in SEGMENTS:
        v    = emb[seg][:20]
        row1 = ", ".join([f"{x:>6.3f}" for x in v[:10]])
        row2 = ", ".join([f"{x:>6.3f}" for x in v[10:20]])
        print()
        print(f"  {seg:<7}: [ {row1},")
        print(f"           {row2} ]")
    print()
    print("Can you spot differences between early, middle, late?")
    print("These small differences = how the voice changed over time.")
    print()

    # ── STEP D ────────────────────────────────────────────────────────────
    print(f"STEP D: EMBEDDING STATISTICS for {orig_stem}:")
    print()
    print(f"  {'Statistic':<14}  {'early':>10}  {'middle':>10}  {'late':>10}")
    print("  " + "-" * 48)
    for label, fn in [
        ("min value",  lambda v: v.min()),
        ("max value",  lambda v: v.max()),
        ("mean value", lambda v: v.mean()),
        ("std dev",    lambda v: v.std()),
        ("num zeros",  lambda v: float((v == 0).sum())),
        ("num > 0",    lambda v: float((v  > 0).sum())),
        ("num < 0",    lambda v: float((v  < 0).sum())),
    ]:
        vals = [fn(emb[s]) for s in SEGMENTS]
        fmt  = "{:>10.3f}" if "num" not in label else "{:>10.0f}"
        row  = "  ".join([fmt.format(v) for v in vals])
        print(f"  {label:<14}  {row}")
    print()
    if all(emb[s].std() < 0.01 for s in SEGMENTS):
        print("  WARNING: std dev near 0 — embedding may be collapsed")
    if any((emb[s] == 0).sum() > 500 for s in SEGMENTS):
        print("  WARNING: many zero features detected")
    print()

    # ── STEP E: 4 plots ───────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle(f"Subject: {display_name}  |  Class: {cls}",
                 fontsize=13, fontweight="bold", y=1.01)
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.50, wspace=0.35)

    # Plot 1 — Raw heatmap
    ax1   = fig.add_subplot(gs[0, 0])
    raw_h = seq["early"].copy()
    ds_note = ""
    if raw_h.shape[0] > 200:
        raw_h   = raw_h[::2, :]
        ds_note = "Display downsampled for visibility"
    im = ax1.imshow(raw_h, aspect="auto", cmap="coolwarm",
                    interpolation="nearest", vmin=-3, vmax=3)
    ax1.set_xlabel("Feature index (0-767)", fontsize=9)
    ax1.set_ylabel("Time step", fontsize=9)
    ax1.set_title(f"{display_name}\nEarly Segment — Raw Encoder Output [T x 768]",
                  fontsize=9, fontweight="bold")
    cb = fig.colorbar(im, ax=ax1, fraction=0.03, pad=0.04)
    cb.set_label("Feature Activation Value", fontsize=8)
    if ds_note:
        ax1.text(0.01, -0.13, ds_note, transform=ax1.transAxes,
                 fontsize=7, color="gray", style="italic")
    ax1.text(0.01, -0.20 if ds_note else -0.13,
             "Each row = 1 time step (~20ms). Each col = 1 of 768 features.  "
             "Red = activated, Blue = suppressed.",
             transform=ax1.transAxes, fontsize=7, color="#555555", style="italic")

    # Plot 2 — Averaged embedding line chart
    ax2 = fig.add_subplot(gs[0, 1])
    for seg in SEGMENTS:
        ax2.plot(emb[seg], alpha=0.75, linewidth=0.7,
                 color=SEG_COLORS[seg], label=seg)
    ax2.set_xlabel("Feature index (0-767)", fontsize=9)
    ax2.set_ylabel("Feature value", fontsize=9)
    ax2.set_title(f"{display_name}\nAveraged Embedding — early vs middle vs late",
                  fontsize=9, fontweight="bold")
    ax2.legend(fontsize=8, loc="upper right")
    ax2.text(0.01, -0.13,
             "Each point = one of 768 features after averaging over time.  "
             "Where lines differ = voice changed between segments.",
             transform=ax2.transAxes, fontsize=7, color="#555555", style="italic")

    # Plot 3 — Difference: |F_late - F_early|
    ax3    = fig.add_subplot(gs[1, 0])
    delta  = np.abs(emb["late"] - emb["early"])
    top15  = set(np.argsort(delta)[-15:])
    c_bars = ["#E53935" if i in top15 else "#BDBDBD" for i in range(len(delta))]
    ax3.bar(range(len(delta)), delta, color=c_bars, linewidth=0)
    ax3.set_xlabel("Feature index", fontsize=9)
    ax3.set_ylabel("Absolute change", fontsize=9)
    ax3.set_title(f"{display_name}\nVoice Change: early -> late (per feature)",
                  fontsize=9, fontweight="bold")
    ax3.text(0.01, -0.13,
             "Red bars = top 15 features that changed most from early to late.  "
             "These carry voice progression information.",
             transform=ax3.transAxes, fontsize=7, color="#555555", style="italic")

    # Plot 4 — Segment similarity bars
    ax4         = fig.add_subplot(gs[1, 1])
    pair_keys   = [("early","middle"), ("middle","late"), ("early","late")]
    pair_labels = ["early<->middle", "middle<->late", "early<->late"]
    pair_colors = [SEG_COLORS["early"], SEG_COLORS["middle"], SEG_COLORS["late"]]
    sims        = [cos_sim(emb[a], emb[b]) for a, b in pair_keys]
    grp_avg     = float(np.mean(sims))
    bars = ax4.bar(pair_labels, sims, color=pair_colors, alpha=0.85, edgecolor="white")
    ax4.axhline(grp_avg, color="black", linestyle="--", linewidth=1,
                label=f"avg = {grp_avg:.3f}")
    for bar, val in zip(bars, sims):
        ax4.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.003,
                 f"{val:.3f}", ha="center", va="bottom",
                 fontsize=9, fontweight="bold")
    ax4.set_ylim(0, 1.08)
    ax4.set_ylabel("Cosine Similarity", fontsize=9)
    ax4.set_title(f"{display_name} | {cls}\nHow Similar Are the 3 Segments?",
                  fontsize=9, fontweight="bold")
    ax4.legend(fontsize=8)
    ax4.text(0.01, -0.13,
             "1.0 = identical voice in both segments.  "
             "Lower = more voice change = potential disease effect.",
             transform=ax4.transAxes, fontsize=7, color="#555555", style="italic")

    plt.tight_layout()
    safe  = orig_stem.replace("/", "_").replace(" ", "_")
    fpath = os.path.join(OUT_DIR, f"{safe}_story.png")
    fig.savefig(fpath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Figure saved -> {fpath}")

    summary_rows.append({
        "subject":        orig_stem,
        "cls":            cls,
        "early_late_sim": cos_sim(emb["early"], emb["late"]),
    })


Processing subject 1/6: 000c901b-049b-41e1-b238-eae9c209edaa_6023402
SUBJECT REPORT: 000c901b-049b-41e1-b238-eae9c209edaa_6023402  |  Class: PD

STEP A: RAW SHAPE BEFORE AVERAGING
Raw encoder output (before averaging):
  early  segment -> shape: [149  time_steps x 768 features]
  middle segment -> shape: [199 time_steps x 768 features]
  late   segment -> shape: [149  time_steps x 768 features]

This means:
  early : encoder listened 149  times (every ~20ms), 768 numbers each time
  middle: encoder listened 199 times
  late  : encoder listened 149  times

STEP B: RAW OUTPUT — first 5 time steps x first 10 features (early segment):
(Each row = one ~20ms chunk of voice. Each column = one feature.)

        feat_0     feat_1     feat_2     feat_3     feat_4     feat_5     feat_6     feat_7     feat_8     feat_9   
time_0:   -0.003     0.135     0.474     0.167    -0.336    -0.099     0.338     2.737     0.041     0.025
time_1:    0.016     0.152     0.476     0.160    -0.318    -0.118   

  Figure saved -> /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/HuBERT/results/subject_stories/000c901b-049b-41e1-b238-eae9c209edaa_6023402_story.png

Processing subject 2/6: 01c1c8cc-cc5b-48a4-b1b8-431065bf780f_6013793
SUBJECT REPORT: 01c1c8cc-cc5b-48a4-b1b8-431065bf780f_6013793  |  Class: PD

STEP A: RAW SHAPE BEFORE AVERAGING
Raw encoder output (before averaging):
  early  segment -> shape: [149  time_steps x 768 features]
  middle segment -> shape: [199 time_steps x 768 features]
  late   segment -> shape: [149  time_steps x 768 features]

This means:
  early : encoder listened 149  times (every ~20ms), 768 numbers each time
  middle: encoder listened 199 times
  late  : encoder listened 149  times

STEP B: RAW OUTPUT — first 5 time steps x first 10 features (early segment):
(Each row = one ~20ms chunk of voice. Each column = one feature.)

        feat_0     feat_1     feat_2     feat_3     feat_4     feat_5     feat_6     feat_7     feat_8     feat_9   
time_0:    0.1

  Figure saved -> /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/HuBERT/results/subject_stories/01c1c8cc-cc5b-48a4-b1b8-431065bf780f_6013793_story.png

Processing subject 3/6: 024dc78c-80c8-4d52-8600-0101d0fa2c83_6012162
SUBJECT REPORT: 024dc78c-80c8-4d52-8600-0101d0fa2c83_6012162  |  Class: PD

STEP A: RAW SHAPE BEFORE AVERAGING
Raw encoder output (before averaging):
  early  segment -> shape: [149  time_steps x 768 features]
  middle segment -> shape: [199 time_steps x 768 features]
  late   segment -> shape: [149  time_steps x 768 features]

This means:
  early : encoder listened 149  times (every ~20ms), 768 numbers each time
  middle: encoder listened 199 times
  late  : encoder listened 149  times

STEP B: RAW OUTPUT — first 5 time steps x first 10 features (early segment):
(Each row = one ~20ms chunk of voice. Each column = one feature.)

        feat_0     feat_1     feat_2     feat_3     feat_4     feat_5     feat_6     feat_7     feat_8     feat_9   
time_0:    0.1

  Figure saved -> /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/HuBERT/results/subject_stories/024dc78c-80c8-4d52-8600-0101d0fa2c83_6012162_story.png

Processing subject 4/6: 003d12eb-7a9c-4ab2-aed8-e6097c5d5c38_6013609
SUBJECT REPORT: 003d12eb-7a9c-4ab2-aed8-e6097c5d5c38_6013609  |  Class: HC

STEP A: RAW SHAPE BEFORE AVERAGING
Raw encoder output (before averaging):
  early  segment -> shape: [149  time_steps x 768 features]
  middle segment -> shape: [199 time_steps x 768 features]
  late   segment -> shape: [149  time_steps x 768 features]

This means:
  early : encoder listened 149  times (every ~20ms), 768 numbers each time
  middle: encoder listened 199 times
  late  : encoder listened 149  times

STEP B: RAW OUTPUT — first 5 time steps x first 10 features (early segment):
(Each row = one ~20ms chunk of voice. Each column = one feature.)

        feat_0     feat_1     feat_2     feat_3     feat_4     feat_5     feat_6     feat_7     feat_8     feat_9   
time_0:   -0.0

  Figure saved -> /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/HuBERT/results/subject_stories/003d12eb-7a9c-4ab2-aed8-e6097c5d5c38_6013609_story.png

Processing subject 5/6: 0055c363-8a10-4081-91e2-dae9d082752c_6011857
SUBJECT REPORT: 0055c363-8a10-4081-91e2-dae9d082752c_6011857  |  Class: HC

STEP A: RAW SHAPE BEFORE AVERAGING
Raw encoder output (before averaging):
  early  segment -> shape: [149  time_steps x 768 features]
  middle segment -> shape: [199 time_steps x 768 features]
  late   segment -> shape: [149  time_steps x 768 features]

This means:
  early : encoder listened 149  times (every ~20ms), 768 numbers each time
  middle: encoder listened 199 times
  late  : encoder listened 149  times

STEP B: RAW OUTPUT — first 5 time steps x first 10 features (early segment):
(Each row = one ~20ms chunk of voice. Each column = one feature.)

        feat_0     feat_1     feat_2     feat_3     feat_4     feat_5     feat_6     feat_7     feat_8     feat_9   
time_0:    0.0

  Figure saved -> /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/HuBERT/results/subject_stories/0055c363-8a10-4081-91e2-dae9d082752c_6011857_story.png

Processing subject 6/6: 00cc4230-4c28-4546-b353-e3ae13a7f0b0_6001885
SUBJECT REPORT: 00cc4230-4c28-4546-b353-e3ae13a7f0b0_6001885  |  Class: HC

STEP A: RAW SHAPE BEFORE AVERAGING
Raw encoder output (before averaging):
  early  segment -> shape: [149  time_steps x 768 features]
  middle segment -> shape: [199 time_steps x 768 features]
  late   segment -> shape: [149  time_steps x 768 features]

This means:
  early : encoder listened 149  times (every ~20ms), 768 numbers each time
  middle: encoder listened 199 times
  late  : encoder listened 149  times

STEP B: RAW OUTPUT — first 5 time steps x first 10 features (early segment):
(Each row = one ~20ms chunk of voice. Each column = one feature.)

        feat_0     feat_1     feat_2     feat_3     feat_4     feat_5     feat_6     feat_7     feat_8     feat_9   
time_0:   -0.0

  Figure saved -> /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/HuBERT/results/subject_stories/00cc4230-4c28-4546-b353-e3ae13a7f0b0_6001885_story.png


In [6]:
print()
print("=" * 65)
print("QUICK COMPARISON: PD vs HC voice change")
print("=" * 65)
print(f"  {'Subject':<35} | {'Class':<6} | {'early<->late':>12} | Change")
print("  " + "-" * 65)

pd_sims, hc_sims = [], []
for row in summary_rows:
    sim   = row["early_late_sim"]
    level = "HIGH" if sim < 0.90 else ("MED" if sim < 0.95 else "LOW")
    lbl   = short(row["subject"], n=33)
    print(f"  {lbl:<35} | {row['cls']:<6} | {sim:>12.4f} | {level}")
    (pd_sims if row["cls"] == "PD" else hc_sims).append(sim)

print()
pd_mean = float(np.mean(pd_sims))
hc_mean = float(np.mean(hc_sims))
print(f"  PD mean early<->late similarity : {pd_mean:.4f}")
print(f"  HC mean early<->late similarity : {hc_mean:.4f}")
print()
diff = abs(pd_mean - hc_mean) * 100
if pd_mean < hc_mean:
    print(f"  Pattern: PD subjects show MORE voice change than HC ({diff:.1f}pp lower similarity)")
    print("  This supports the value of temporal segmentation for PD detection.")
elif pd_mean > hc_mean:
    print(f"  Pattern: PD subjects show LESS voice change than HC ({diff:.1f}pp higher similarity)")
    print("  PD voices may be more monotone/uniform — also a discriminative signal.")
else:
    print("  Pattern: PD and HC show SIMILAR voice change across segments.")
print()
print(f"  All figures saved to: {OUT_DIR}/")


QUICK COMPARISON: PD vs HC voice change
  Subject                             | Class  | early<->late | Change
  -----------------------------------------------------------------
  000c901b-049b-41e1-b238-eae9c209e... | PD     |       0.8509 | HIGH
  01c1c8cc-cc5b-48a4-b1b8-431065bf7... | PD     |       0.9354 | MED
  024dc78c-80c8-4d52-8600-0101d0fa2... | PD     |       0.8944 | HIGH
  003d12eb-7a9c-4ab2-aed8-e6097c5d5... | HC     |       0.8698 | HIGH
  0055c363-8a10-4081-91e2-dae9d0827... | HC     |       0.8631 | HIGH
  00cc4230-4c28-4546-b353-e3ae13a7f... | HC     |       0.9240 | MED

  PD mean early<->late similarity : 0.8936
  HC mean early<->late similarity : 0.8856

  Pattern: PD subjects show LESS voice change than HC (0.8pp higher similarity)
  PD voices may be more monotone/uniform — also a discriminative signal.

  All figures saved to: /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/HuBERT/results/subject_stories/
